# Unit 9 / Chapter 9: Quantum Generative Models

> **Main Learning Objective:** Understand what generative models do in modern AI, see how quantum computers are natural sampling machines, and build a small Born machine that learns to reproduce a target distribution.

| Section | Topic | Why it matters for AI |
|---|---|---|
| 9.1 | Generative models today | GANs, VAEs, diffusion, LLMs all learn distributions |
| 9.2 | Quantum states are natural samplers | Measurement = draw from |amplitude|^2 |
| 9.3 | Born machines | Learn a distribution by tuning circuit parameters |
| 9.4 | Quantum GANs and diffusion | Cutting edge research directions |

By the end of this unit you should be able to:

1. Explain what a generative model does.
2. Describe how measurement of a quantum state is exactly sampling from a probability distribution.
3. Train a tiny Born machine to fit a 1D target distribution.
4. Recognize when a Quantum GAN or Quantum diffusion model is being described.
5. Identify at least one real world domain where quantum generative models could plausibly matter.

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100
print("Imports done.")

---
## Course check-in

This logs that you started **Unit 9**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 9
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 9.1: Generative Models Today

Generative models learn a probability distribution p(x) over data and can then produce new samples that look like the training data.

| Family | Trained by | Famous example |
|---|---|---|
| GAN | Two networks in a game: generator vs discriminator | StyleGAN faces |
| VAE | Encoder + decoder + KL constraint | Older image models |
| Normalizing flow | Invertible neural network | RealNVP, Glow |
| Autoregressive | Predict next token given previous | GPT, LLaMA, Claude |
| Diffusion | Learn to denoise a noisy image step by step | Stable Diffusion, DALL-E 3 |

All of them ultimately do one thing: sample from a learned distribution p(x). Some domains where this matters: image generation, text generation, molecular design, audio synthesis, and Monte Carlo simulation.

In [ ]:
# Illustrate the core idea: fit a mixture of 2 Gaussians and draw samples.
xs = np.linspace(-6, 6, 500)
true = 0.5*np.exp(-0.5*((xs+2)/0.8)**2) + 0.5*np.exp(-0.5*((xs-2)/1.2)**2)
true /= true.sum()

samples = np.random.choice(xs, size=3000, p=true)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(xs, true, color='#5B2C91', lw=2)
axes[0].fill_between(xs, true, alpha=0.2, color='#5B2C91')
axes[0].set_title('Target distribution p(x)')
axes[1].hist(samples, bins=60, density=True, color='#2A9D8F', alpha=0.8)
axes[1].plot(xs, true / (xs[1]-xs[0]) / xs.size * 1000, color='black', lw=1.2)
axes[1].set_title('3000 samples drawn from p(x)')
plt.tight_layout(); plt.show()

---
# Section 9.2: Quantum States are Natural Samplers

When you measure a quantum state |psi> = sum_i alpha_i |i>, you get outcome i with probability |alpha_i|^2. So measurement IS drawing a sample from a probability distribution.

That means every quantum circuit is a sampler for some distribution over bit strings. The distribution is called the Born distribution, and a trainable circuit that produces it is called a **Born machine**.

Key advantages:

- Naturally represents an exponentially large discrete distribution using only log(N) qubits.
- Interference lets the model concentrate probability in specific regions.
- No need for an explicit density function; you can train just from samples.

Key limitations:

- Reading many samples from a real quantum device is slow.
- Not all classical distributions are easy to represent on a shallow quantum circuit.
- Training is prone to barren plateaus.

In [ ]:
# The measurement histogram from a small circuit IS a sample from a distribution.
# Take a 3-qubit random state and show its Born distribution.
np.random.seed(2)
alpha = np.random.randn(8) + 1j * np.random.randn(8)
alpha /= np.linalg.norm(alpha)
probs = np.abs(alpha)**2

labels = [f"|{i:03b}>" for i in range(8)]
plt.bar(labels, probs, color='#5B2C91')
plt.title('Born distribution of a random 3-qubit state')
plt.ylabel('P(outcome)')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

samples = np.random.choice(8, size=5000, p=probs)
hist = np.bincount(samples, minlength=8) / 5000
print("Expected vs measured on 5000 samples:")
for i, (p, h) in enumerate(zip(probs, hist)):
    print(f"  |{i:03b}>: expected {p:.3f}, measured {h:.3f}")

---
# Section 9.3: Train a Born Machine

The Born machine has trainable parameters theta. We measure the circuit many times to get samples, compare their distribution to the target, and update theta to shrink the mismatch.

Below we train a tiny 2 qubit Born machine to match a target distribution over the 4 outcomes {00, 01, 10, 11}.

In [ ]:
# Tiny 2-qubit ansatz: two Ry rotations, one CNOT, two more Ry rotations
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]])
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])

def born_probs(theta):
    # theta has 4 entries
    op = np.kron(Ry(theta[0]), Ry(theta[1]))
    op = CNOT @ op
    op2 = np.kron(Ry(theta[2]), Ry(theta[3]))
    op = op2 @ op
    psi = op @ np.array([1.0, 0, 0, 0])
    return psi**2  # real amplitudes -> just square

def kl(p, q, eps=1e-9):
    return np.sum(p * np.log((p+eps)/(q+eps)))

# Target: peaked on |01> and |10> (like a small Bell-ish distribution)
target = np.array([0.05, 0.45, 0.45, 0.05])

theta = np.random.uniform(-1, 1, 4)
lr = 0.5
history = []
for step in range(120):
    p = born_probs(theta)
    loss = kl(target, p)
    history.append(loss)

    # numeric gradient
    grad = np.zeros_like(theta)
    for i in range(4):
        eps = 1e-3
        t1 = theta.copy(); t1[i] += eps
        t2 = theta.copy(); t2[i] -= eps
        grad[i] = (kl(target, born_probs(t1)) - kl(target, born_probs(t2))) / (2*eps)
    theta = theta - lr * grad

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history, color='#5B2C91', lw=2)
axes[0].set_xlabel('step'); axes[0].set_ylabel('KL(target || model)')
axes[0].set_title('Born machine training loss')
axes[0].grid(alpha=0.3)

labels = ['|00>','|01>','|10>','|11>']
x = np.arange(4)
axes[1].bar(x-0.18, target,           width=0.35, label='target',   color='#5B2C91')
axes[1].bar(x+0.18, born_probs(theta), width=0.35, label='learned', color='#2A9D8F')
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel('P'); axes[1].legend(); axes[1].set_title('After training')
plt.tight_layout(); plt.show()

---
# Section 9.4: Quantum GANs and Diffusion

Modern quantum generative research goes beyond plain Born machines:

- **Quantum GAN (QGAN):** a generator quantum circuit and a discriminator (classical or quantum) play the same adversarial game as classical GANs.
- **Quantum diffusion models:** noise is added and then removed step by step, but the removal step is done by a quantum circuit.
- **Quantum autoregressive models:** predict the next qubit given previous measurements.

None of these have shown a clear quantum advantage yet. But they are among the most active research areas because generative models drive so much of modern AI.

## Where they could matter first

- **Molecule generation** for drug discovery (where quantum-native chemistry structure helps).
- **Small combinatorial distributions** like feature selection masks or scheduling assignments.
- **Financial simulation** for Monte Carlo pricing of exotic derivatives.

---
# End-of-Unit Quiz

Ten multiple choice questions. Reveal the answer to check yourself.

**1. A generative model learns to:**

A) Predict labels for input data
B) Sample from a distribution p(x) over data
C) Compress data to lower dimensions
D) Sort a list of numbers

<details><summary>Answer 1</summary>**B).** Generative models model and sample from data distributions.</details>

---

**2. Which famous model family is essentially generative and autoregressive?**

A) Support Vector Machine
B) K-means
C) GPT / LLaMA / Claude (LLMs)
D) PCA

<details><summary>Answer 2</summary>**C).** LLMs predict the next token given the previous ones.</details>

---

**3. When you measure a quantum state, you get outcome i with probability:**

A) alpha_i
B) |alpha_i|
C) |alpha_i|^2
D) always 1/N

<details><summary>Answer 3</summary>**C) |alpha_i|^2.** This is the Born rule.</details>

---

**4. A Born machine is:**

A) A quantum device that only outputs |0>
B) A trainable quantum circuit whose measurement distribution learns a target
C) A classical GAN with quantum weights
D) A quantum error correcting code

<details><summary>Answer 4</summary>**B).** The circuit's measurement distribution is the model.</details>

---

**5. What is a quantum GAN's generator?**

A) A neural network
B) A parameterized quantum circuit that produces samples
C) A random noise source
D) A discriminator

<details><summary>Answer 5</summary>**B).** The generator is a PQC in a QGAN.</details>

---

**6. Which of these is NOT a family of generative models?**

A) GAN
B) VAE
C) K-Nearest Neighbors
D) Diffusion

<details><summary>Answer 6</summary>**C) K-Nearest Neighbors.** K-NN is a discriminative classifier, not a generative model.</details>

---

**7. What does the KL divergence KL(p || q) measure?**

A) The gradient of a classifier
B) A distance between two probability distributions
C) The similarity of two vectors
D) The variance of a dataset

<details><summary>Answer 7</summary>**B).** KL divergence quantifies how one distribution differs from another and is the standard loss for maximum-likelihood training.</details>

---

**8. Why are Born machines called exponentially compact?**

A) They compress the training data
B) They use only log_2(N) qubits to hold an N-dimensional discrete distribution
C) They need fewer training epochs
D) They use less RAM than neural networks

<details><summary>Answer 8</summary>**B).** N amplitudes are stored in n = log_2(N) qubits.</details>

---

**9. What role does the discriminator play in a Quantum GAN?**

A) It generates fake samples
B) It distinguishes real from generated samples and pushes the generator to improve
C) It corrects quantum errors
D) It initializes the qubits

<details><summary>Answer 9</summary>**B).** Just like classical GANs, the discriminator provides the adversarial signal.</details>

---

**10. Which domain most naturally fits quantum generative models today?**

A) Sorting numbers
B) Molecule generation for drug discovery
C) Plotting histograms
D) Computing matrix inverses

<details><summary>Answer 10</summary>**B).** Molecules are naturally quantum objects with rich discrete structure, matching the strengths of quantum generative models.</details>

---

## Unit 9 Summary

| Concept | Key idea |
|---|---|
| Generative model | Learn a distribution p(x) and produce fresh samples. |
| Born rule | Measurement outcome distribution is |amplitude|^2. |
| Born machine | Trainable quantum circuit whose measurement distribution is the model. |
| KL divergence | Standard loss for training a distribution to match a target. |
| Quantum GAN | Adversarial training with a quantum generator. |
| Quantum diffusion | Denoising with a quantum circuit at each step. |
| Applications | Molecule design, materials, finance, combinatorial optimization. |

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 9!")